# Module 7 — Oak Ridge Reservoir contributing watershed and USGS stations

Adapted from Module 6 with outlet **USGS 01382200**, Oak Ridge Reservoir at Oak Ridge, NJ.

**Method:** retrieve the USGS NLDI unsimplified upstream basin associated with the reservoir station. This is an NHDPlus-based basin retrieval, not a fresh DEM delineation or a surveyed boundary. The contributing watershed is not the reservoir water surface or a larger HUC watershed.

**Area check:** compare the retrieved basin with the USGS station drainage area of **27.3 mi²**. The notebook reports the actual difference and does not force a match. Surface drainage does not describe reservoir diversions or operational transfers.

**Station scope:** all NWIS site types and statuses, with a **2 km outward buffer**, matching Module 6. Exact geometric filtering follows the bounding-box query. Boundary points count as in-basin. Nearby buffer-only sites are not necessarily hydrologically connected.

Station counts and basin area are calculated on execution. Station presence does not imply current operation or availability of precipitation, discharge, or stage. IDs remain strings.

Sources: [USGS outlet station](https://waterdata.usgs.gov/monitoring-location/USGS-01382200/), [Module 6](https://github.com/JohannVera/newark-water-supply/blob/main/notebooks/Module6_CanistearWatershed_USGS.ipynb). Exact retrieval URLs and UTC timestamps are saved in `data/oak_ridge/metadata.json`, with raw responses in `sources/`.


## 1. Environment
Run this notebook from its module folder. Python packages: requests, pandas, geopandas, shapely, pyproj, pyogrio. In Colab or a new environment, uncomment the install line below. QGIS export is optional on rerun; a validated QGIS project is already supplied.

## Reuse this workflow
Edit the study configuration, then run cells in order. Output files go to `data/oak_ridge`.

The 2 km buffer is a proximity search inherited from Module 6, not a hydraulic connection or a travel-time criterion. Use a local projected metric CRS for another region. Review the area discrepancy and station catalog before engineering use.


In [ ]:
# %pip install requests pandas geopandas shapely pyproj pyogrio
from pathlib import Path


## 2. Retrieve basin, build buffer, discover all stations, and export
EPSG:32618 (UTM zone 18N) is used for the 2,000 m buffer, distance, and area operations. GeoJSON and interactive map coordinates are WGS84. GeoPackage layers retain UTM coordinates. The public NWIS query covers the buffer bounding box, then exact geometric filtering removes outside points. Failed requests raise errors rather than silently generating a zero-station result.

The HTML map uses Leaflet with the same Esri World Street Map provider as Module 5. Internet is required for JavaScript and basemap tiles; the GeoPackage and QGIS vector layers work offline.

### Imports

In [ ]:
from pathlib import Path
import json, io, datetime, html
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape



### Study configuration

In [ ]:
# Reusable study settings: change these for another USGS outlet.
OUTLET_SITE = '01382200'
STUDY_NAME = 'Oak Ridge Reservoir contributing watershed'
BUFFER_DISTANCE_KM = 2.0
PROJECTED_CRS = 32618  # UTM 18N; select the local metric CRS for another region.
PUBLISHED_AREA_SQMI = 27.3  # Optional independent area check; update for another site.



### Output folder and public-service request helper

In [ ]:
# Locate the project root in a clone; Colab defaults to its current working directory.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'oak_ridge').is_dir()), Path.cwd())
OUT = ROOT / 'data' / 'oak_ridge'
OUT.mkdir(parents=True, exist_ok=True)
RAW = OUT / 'sources'; RAW.mkdir(exist_ok=True)
session = requests.Session()
log = []
def fetch(url, params, name):
    r = session.get(url, params=params, timeout=90); r.raise_for_status()
    (RAW / name).write_text(r.text, encoding='utf-8')
    log.append({'url':r.url,'retrieved_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'file':name})
    return r



### Retrieve the contributing watershed and check its area

In [ ]:
basin_json = fetch(f'https://api.water.usgs.gov/nldi/linked-data/nwissite/USGS-{OUTLET_SITE}/basin', {'simplified':'false'}, 'nldi_basin.geojson').json()
basin = gpd.GeoDataFrame.from_features(basin_json['features'], crs=4326).to_crs(PROJECTED_CRS)
basin = gpd.GeoDataFrame({'name':[STUDY_NAME]}, geometry=[basin.geometry.union_all()], crs=PROJECTED_CRS)
area = basin.area.iloc[0]/1e6
basin['area_km2'] = area; basin['area_sqmi'] = area/2.589988110336

if PUBLISHED_AREA_SQMI:
    area_difference_pct = 100 * (area / 2.589988110336 - PUBLISHED_AREA_SQMI) / PUBLISHED_AREA_SQMI
    print(f'NLDI area: {area:.2f} km²; published USGS: {PUBLISHED_AREA_SQMI:g} mi²; difference: {area_difference_pct:+.2f}%')


### Create the metric search buffer

In [ ]:
buffer = basin.copy(); buffer['name'] = f'{BUFFER_DISTANCE_KM:g} km watershed search buffer'; buffer.geometry = basin.buffer(BUFFER_DISTANCE_KM * 1000)
ring = buffer.copy(); ring['name'] = f'{BUFFER_DISTANCE_KM:g} km buffer outside watershed'; ring.geometry = buffer.geometry.difference(basin.geometry.iloc[0])


### Retrieve the USGS station inventory

In [ ]:
w,s,e,n = buffer.to_crs(4326).total_bounds
params = {'format':'rdb','bBox':f'{w:.7f},{s:.7f},{e:.7f},{n:.7f}','siteStatus':'all','siteOutput':'expanded'}
r = fetch('https://waterservices.usgs.gov/nwis/site/', params, 'nwis_sites.rdb')
lines = [x for x in r.text.splitlines() if x and not x.startswith('#')]
df = pd.read_csv(io.StringIO('\n'.join([lines[0]]+lines[2:])), sep='\t', dtype=str).fillna('')
assert len(df)>0, 'No station response; do not produce an empty catalog.'
df['dec_lat_va'] = pd.to_numeric(df.dec_lat_va, errors='coerce'); df['dec_long_va'] = pd.to_numeric(df.dec_long_va, errors='coerce')
assert df[['dec_lat_va','dec_long_va']].notna().all().all(), 'Missing coordinates require review.'


### Filter, classify, and calculate distances

In [ ]:
sites = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.dec_long_va,df.dec_lat_va), crs=4326).to_crs(PROJECTED_CRS)
sites = sites[sites.geometry.intersects(buffer.geometry.iloc[0])].copy().drop_duplicates('site_no')
sites['location_type'] = ['watershed' if basin.geometry.iloc[0].covers(p) else 'buffer_only' for p in sites.geometry]
sites['distance_to_basin_km'] = sites.distance(basin.geometry.iloc[0])/1000
sites['distance_to_centroid_km'] = sites.distance(basin.geometry.iloc[0].centroid)/1000
sites['usgs_url'] = 'https://waterdata.usgs.gov/monitoring-location/USGS-'+sites.site_no+'/'
sites = sites.sort_values(['location_type','site_no'])

assert OUTLET_SITE in set(sites.site_no), 'Outlet absent from exact station selection; review NLDI snap and coordinates.'


### Export GIS layers, catalog, and provenance

In [ ]:
layers = {'watershed':basin,'search_buffer':buffer,'buffer_only_zone':ring,'usgs_sites':sites,
          'sites_in_basin':sites[sites.location_type=='watershed'].copy(), 'sites_buffer_only':sites[sites.location_type=='buffer_only'].copy()}
gpkg = OUT/'OakRidge.gpkg'
if gpkg.exists(): gpkg.unlink()
for name, frame in layers.items():
    frame.to_file(gpkg,layer=name,driver='GPKG')
    frame.to_crs(4326).to_file(OUT/(name+'.geojson'),driver='GeoJSON')
sites.drop(columns='geometry').to_csv(OUT/'usgs_station_catalog.csv',index=False)
summary = {'watershed_area_km2':area,'watershed_area_sqmi':area/2.589988110336,'published_usgs_area_sqmi':PUBLISHED_AREA_SQMI,
 'basin_sites':int((sites.location_type=='watershed').sum()),'buffer_only_sites':int((sites.location_type=='buffer_only').sum()),
 'total_sites':len(sites),'buffer_km':BUFFER_DISTANCE_KM,'crs':f'EPSG:{PROJECTED_CRS}','method':f'USGS NLDI unsimplified upstream basin for USGS-{OUTLET_SITE}; not a new DEM delineation',
 'scope':'All NWIS site types, all statuses, no parameter filter; exact polygon intersection after bounding-box query',
 'area_difference_pct':area_difference_pct, 'sources':log}
(OUT/'metadata.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
print(json.dumps({k:v for k,v in summary.items() if k!='sources'},indent=2))



### Create the interactive map

In [ ]:
# Module 5-style interactive map, with separate station groups and USGS popups.
data = {k:json.loads(v.to_crs(4326).to_json()) for k,v in layers.items() if k in ['watershed','search_buffer','sites_in_basin','sites_buffer_only']}
page = '''<!doctype html><html><head><meta charset="utf-8"><title>Oak Ridge Reservoir watershed and USGS stations</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"><script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<style>body{margin:0;font:15px Arial}#map{height:88vh}header{padding:12px 20px}h2{margin:0 0 6px} .legend{background:white;padding:12px;line-height:1.6}</style></head><body>
<header><h2>Oak Ridge Reservoir contributing watershed</h2>AREA km² · INSIDE sites in basin · OUTSIDE sites in BUFFER_LABEL buffer only · All USGS site types and statuses</header><div id="map"></div>
<script>const data=DATA; const map=L.map('map');
L.tileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}',{attribution:'Tiles © Esri and the GIS User Community'}).addTo(map);
const buf=L.geoJSON(data.search_buffer,{style:{color:'#3388cc',weight:1,fillOpacity:.16}}).addTo(map);
const basin=L.geoJSON(data.watershed,{style:{color:'#173d89',weight:3,fillOpacity:.35}}).addTo(map);
function safe(s){const d=document.createElement('div');d.textContent=s||'';return d.innerHTML}
function points(fc,color){return L.geoJSON(fc,{pointToLayer:(f,ll)=>L.circleMarker(ll,{radius:5,color:'#fff',weight:1,fillColor:color,fillOpacity:.9}),onEachFeature:(f,l)=>{const p=f.properties;l.bindTooltip(p.site_no+' '+p.station_nm);l.bindPopup('<b>USGS '+safe(p.site_no)+'</b><br>'+safe(p.station_nm)+'<br>Type: '+safe(p.site_tp_cd)+'<br>Zone: '+safe(p.location_type)+'<br>Distance to basin: '+Number(p.distance_to_basin_km).toFixed(2)+' km<br><a target="_blank" rel="noopener" href="'+p.usgs_url+'">USGS station page</a>')}}).addTo(map)}
const inside=points(data.sites_in_basin,'#16834b'),outside=points(data.sites_buffer_only,'#ed8b23');
L.control.layers(null,{'BUFFER_LABEL search buffer':buf,'Contributing watershed':basin,'USGS sites in basin':inside,'USGS sites in buffer only':outside}).addTo(map);
map.fitBounds(buf.getBounds());const legend=L.control({position:'bottomleft'});legend.onAdd=()=>{const d=L.DomUtil.create('div','legend');d.innerHTML='<span style="color:#16834b">●</span> USGS site in basin<br><span style="color:#ed8b23">●</span> USGS site in buffer only<br>Blue: watershed and BUFFER_LABEL buffer<br>Source: USGS NLDI / NWIS';return d};legend.addTo(map);</script></body></html>'''
page = page.replace('Oak Ridge Reservoir contributing watershed', html.escape(STUDY_NAME)).replace('BUFFER_LABEL', f'{BUFFER_DISTANCE_KM:g} km').replace('AREA',f'{area:.2f}').replace('INSIDE',str(summary['basin_sites'])).replace('OUTSIDE',str(summary['buffer_only_sites'])).replace('DATA',json.dumps(data))
(OUT/'OakRidge_map.html').write_text(page,encoding='utf-8')



## 3. View the supplied station catalog and interactive map
This cell works without repeating the web queries. The CSV site numbers must be read as text.

In [ ]:
import pandas as pd
from IPython.display import display, HTML
import html, json
catalog = pd.read_csv(OUT / 'usgs_station_catalog.csv', dtype={'site_no': str})
display(catalog[['site_no', 'station_nm', 'site_tp_cd', 'location_type']])
page = (OUT / 'OakRidge_map.html').read_text(encoding='utf-8')
page = page.replace("f'{BUFFER_DISTANCE_KM:g} km search buffer'", json.dumps(f'{BUFFER_DISTANCE_KM:g} km search buffer'))
page = page.replace('Â·', '·')
_ = (OUT / 'OakRidge_map.html').write_text(page, encoding='utf-8')
display(HTML('<iframe style="width:100%;height:750px;border:0" srcdoc="' + html.escape(page, quote=True) + '"></iframe>'))


## 4. Optional QGIS project and PNG export
Run this cell in QGIS Python, after running the setup and data cells in the same session. Colab does not include QGIS. Call `make_qgis()` to create `OakRidge.qgs` and an offline vector PNG. This export is optional and has not been executed in this environment. The GeoPackage can be opened directly in QGIS.


In [ ]:
def make_qgis():
    import os
    dll_handles = []
    if os.name == 'nt':
        for directory in ['C:/Program Files/QGIS 3.44.1/bin','C:/Program Files/QGIS 3.44.1/apps/Qt5/bin','C:/Program Files/QGIS 3.44.1/apps/qgis/bin']:
            if Path(directory).exists(): dll_handles.append(os.add_dll_directory(directory))
    from qgis.core import (QgsApplication,QgsProject,QgsVectorLayer,QgsCoordinateReferenceSystem,QgsFillSymbol,QgsMarkerSymbol,QgsRasterLayer,QgsRectangle,QgsMapSettings,QgsMapRendererParallelJob)
    from qgis.PyQt.QtCore import QSize
    from qgis.PyQt.QtGui import QColor
    app=QgsApplication.instance()
    if app is None:
        app=QgsApplication([],False);app.initQgis()
    project=QgsProject.instance();project.clear();project.setCrs(QgsCoordinateReferenceSystem(f'EPSG:{PROJECTED_CRS}'))
    project.setFileName(str(OUT/'OakRidge.qgs'))
    project.setTitle('Oak Ridge Reservoir contributing watershed and all USGS sites')
    basemap=QgsRasterLayer('type=xyz&url=https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/%7Bz%7D/%7By%7D/%7Bx%7D','Esri World Street Map (internet required)','wms')
    project.addMapLayer(basemap)
    qlayers=[]
    styles=[('search_buffer',f'{BUFFER_DISTANCE_KM:g} km search buffer','110,180,235,45','50,110,190,255'),('watershed',STUDY_NAME,'30,65,145,95','20,50,125,255'),('sites_buffer_only',f'USGS buffer only ({summary["buffer_only_sites"]})','#ed8b23',None),('sites_in_basin',f'USGS in basin ({summary["basin_sites"]})','#16834b',None)]
    for key,title,fill,stroke in styles:
        layer=QgsVectorLayer(str(gpkg)+'|layername='+key,title,'ogr');assert layer.isValid(),key
        symbol=QgsFillSymbol.createSimple({'color':fill,'outline_color':stroke,'outline_width':'0.6'}) if stroke else QgsMarkerSymbol.createSimple({'name':'circle','color':fill,'outline_color':'white','size':'2.5'})
        layer.renderer().setSymbol(symbol);project.addMapLayer(layer);qlayers.insert(0,layer)
    extent=QgsRectangle(*buffer.total_bounds);extent.scale(1.06)
    project.viewSettings().setDefaultViewExtent(__import__('qgis.core',fromlist=['QgsReferencedRectangle']).QgsReferencedRectangle(extent,project.crs()))
    assert project.write(), 'Project save failed'
    # Export an offline map: vector data remain readable without external tiles.
    settings=QgsMapSettings();settings.setLayers(qlayers);settings.setDestinationCrs(project.crs());settings.setExtent(extent);settings.setOutputSize(QSize(1600,1400));settings.setBackgroundColor(QColor('#f4f7fa'))
    job=QgsMapRendererParallelJob(settings);job.start();job.waitForFinished();job.renderedImage().save(str(OUT/'OakRidge_map.png'))
    # Reopen to verify relative paths and layer validity.
    project.clear();assert project.read(str(OUT/'OakRidge.qgs'))
    assert all(x.isValid() for x in project.mapLayers().values()), 'Invalid project layer'
    print('QGIS project saved and reopened; all layers valid.')


# make_qgis()  # Uncomment only in a QGIS Python environment.


## 5. Outputs and interpretation
Open `data/oak_ridge/OakRidge_map.html` for the interactive map (internet required for Leaflet and Esri tiles). Green points are basin sites and orange points are buffer-only sites. Overlapping reservoir/rain gage sites can hide each other at small scales; use the catalog and zoom in.

`OakRidge.gpkg` stores the basin, search buffer, buffer-only zone, and station layers. `usgs_station_catalog.csv` includes IDs, names, zones, distances and station-page links. GeoJSON copies use WGS84. Metric operations and GeoPackage layers use UTM 18N.

For optional QGIS export, run the QGIS cell and keep `OakRidge.qgs` beside `OakRidge.gpkg`. The project uses relative paths.

Review `metadata.json` for the area comparison and provenance. Results represent a public inventory snapshot. No station count, data availability or hydrologic connectivity is assumed from geographic inclusion.


## Validated run snapshot
Executed retrieval and GIS exports on 2026-10-15T22:47:20.167694+00:00.

- Basin area: 69.73 km² (26.92 mi²).
- Published USGS drainage area: 27.3 mi². Difference: -1.37%.
- Stations inside watershed: 7. Buffer-only stations: 10. Total: 17.

The geospatial retrieval, spatial filtering, GeoPackage/GeoJSON/CSV exports and HTML generation completed successfully. The optional QGIS cell has not been run here. Rerunning may change inventory counts. This snapshot is documentation of the validated run, not a cached execution of every notebook cell.
